# Tarea 4: Modelado predictivo supervisado

Objetivo: predecir `TARGET` (probabilidad de default) sobre `application_{train,test}` usando las features construidas en la Tarea 2.

**Plan:**

1. Cargar `features.parquet` y separar train (con `TARGET` conocido) de test (Kaggle, sin `TARGET`).
2. Split train/validation estratificado (el test de Kaggle no tiene etiquetas, así que la validación interna es la única forma de medir performance antes de entregar).
3. Baseline: Regresión Logística (interpretable por diseño, referencia de piso).
4. Modelo primario: XGBoost con manejo de desbalanceo (`scale_pos_weight`), early stopping contra el set de validación.
5. Validación cruzada estratificada (5-fold) sobre el set de train completo para un score de AUC robusto, evitando optimizar contra un único holdout.
6. Métricas: ROC-AUC (primaria, target ≥0.79), PR-AUC, KS statistic, Brier score (calibración, target ≤0.15).
7. Tracking de experimentos con MLflow (params, métricas, artefactos).
8. Feature importance de XGBoost, como insumo para la Tarea 5 (SHAP).

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "../data/processed"

features = pd.read_parquet(f"{DATA_DIR}/features.parquet")
print(features.shape)
print(features["IS_TRAIN"].value_counts())

(356255, 304)
IS_TRAIN
1    307511
0     48744
Name: count, dtype: int64


## Separación de columnas y split train/validation

`SK_ID_CURR` es un identificador, no una feature. `IS_TRAIN` es la bandera que separa train de test de Kaggle, tampoco es una feature del modelo. `TARGET` es la etiqueta.

El set de test de Kaggle (`IS_TRAIN == 0`) no trae `TARGET`: es exclusivamente para generar el submission del leaderboard, no lo podemos usar para medir performance. Por eso separamos, dentro del set de train, un holdout de validación estratificado (mantiene la proporción ~8.07% de default en ambos splits) para poder medir AUC/PR-AUC/KS/Brier antes de tocar el test de Kaggle, evitando así optimizar hiperparámetros contra un set sin etiquetas.

In [2]:
from sklearn.model_selection import train_test_split

NON_FEATURE_COLS = ["SK_ID_CURR", "TARGET", "IS_TRAIN"]

train_full = features.loc[features["IS_TRAIN"] == 1].copy()
kaggle_test = features.loc[features["IS_TRAIN"] == 0].copy()

feature_cols = [c for c in features.columns if c not in NON_FEATURE_COLS]
print(f"Cantidad de features: {len(feature_cols)}")

X = train_full[feature_cols]
y = train_full["TARGET"].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}, tasa de default: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape}, tasa de default: {y_val.mean():.4f}")

Cantidad de features: 301
Train: (246008, 301), tasa de default: 0.0807
Val:   (61503, 301), tasa de default: 0.0807


## Función de métricas

Definimos una única función de evaluación para reutilizar entre el baseline y el modelo primario, con las 4 métricas de evaluación: ROC-AUC (primaria), PR-AUC (relevante por el desbalanceo del 8.07%), KS statistic (estándar en scoring crediticio, mide la máxima separación entre las distribuciones acumuladas de buenos y malos pagadores) y Brier score (calibración: qué tan confiables son las probabilidades predichas, no solo el ranking).

In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve


def evaluate(y_true, y_proba, label):
    auc = roc_auc_score(y_true, y_proba)
    pr_auc = average_precision_score(y_true, y_proba)
    brier = brier_score_loss(y_true, y_proba)
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    ks = np.max(tpr - fpr)

    metrics = {"roc_auc": auc, "pr_auc": pr_auc, "ks": ks, "brier": brier}
    print(f"[{label}] ROC-AUC={auc:.4f}  PR-AUC={pr_auc:.4f}  KS={ks:.4f}  Brier={brier:.4f}")
    return metrics

## Baseline: Regresión Logística

Sirve como piso de referencia interpretable por diseño (coeficientes lineales), antes de justificar la complejidad adicional de XGBoost. Requiere imputación (mediana, mismo criterio que en la Tarea 3) y escalado (`StandardScaler`), a diferencia de XGBoost que maneja NaN nativamente y es invariante a la escala. Usamos `class_weight="balanced"` para compensar el desbalanceo (8.07% de positivos) sin necesidad de submuestrear.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

logreg_pipeline.fit(X_train, y_train)

logreg_val_proba = logreg_pipeline.predict_proba(X_val)[:, 1]
logreg_metrics = evaluate(y_val, logreg_val_proba, "LogReg (val)")

[LogReg (val)] ROC-AUC=0.7637  PR-AUC=0.2488  KS=0.3982  Brier=0.1963


## Modelo primario: XGBoost

XGBoost maneja NaN nativamente (no requiere imputación) y es invariante a la escala de las features, por lo que entrena directamente sobre `X_train`/`X_val` tal como están. Usamos `scale_pos_weight = negativos/positivos` para compensar el desbalanceo del 8.07% (equivalente en gradient boosting al `class_weight="balanced"` de scikit-learn). Los hiperparámetros de árbol (`max_depth`, `learning_rate`, `subsample`, `colsample_bytree`) parten de valores conservadores estándar para Home Credit, con `early_stopping_rounds` contra el set de validación para no sobreajustar en la cantidad de árboles.

In [5]:
import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

xgb_params = dict(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=20,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
)

xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

print(f"Mejor iteración: {xgb_model.best_iteration}")

xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_metrics = evaluate(y_val, xgb_val_proba, "XGBoost (val)")

scale_pos_weight: 11.3871
[0]	validation_0-auc:0.68291
[50]	validation_0-auc:0.74876
[100]	validation_0-auc:0.76047
[150]	validation_0-auc:0.76666
[200]	validation_0-auc:0.77026
[250]	validation_0-auc:0.77280
[300]	validation_0-auc:0.77425
[350]	validation_0-auc:0.77562
[400]	validation_0-auc:0.77657
[450]	validation_0-auc:0.77717
[500]	validation_0-auc:0.77795
[550]	validation_0-auc:0.77860
[600]	validation_0-auc:0.77909
[650]	validation_0-auc:0.77950
[700]	validation_0-auc:0.77976
[750]	validation_0-auc:0.77999
[800]	validation_0-auc:0.78013
[846]	validation_0-auc:0.78009
Mejor iteración: 796
[XGBoost (val)] ROC-AUC=0.7802  PR-AUC=0.2777  KS=0.4241  Brier=0.1751


## Ajuste: XGBoost sin `scale_pos_weight`

El AUC (0.7802) quedó apenas debajo del target (≥0.79) y el Brier (0.1751) por encima del target de calibración (≤0.15). `scale_pos_weight` reescala el gradiente para que los positivos "pesen" ~11x más, lo que empuja las probabilidades predichas hacia arriba de forma sistemática (útil para el ranking en algunos casos, pero a costa de la calibración). Hipótesis: como XGBoost con log-loss ya optimiza directamente sobre probabilidades bien calibradas, sacar `scale_pos_weight` debería mejorar el Brier sin perjudicar (o incluso mejorando levemente) el AUC, que es una métrica de ranking y no de escala absoluta.

In [6]:
xgb_params_nosw = dict(xgb_params)
xgb_params_nosw["scale_pos_weight"] = 1.0

xgb_model_nosw = xgb.XGBClassifier(**xgb_params_nosw)
xgb_model_nosw.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

print(f"Mejor iteración: {xgb_model_nosw.best_iteration}")

xgb_nosw_val_proba = xgb_model_nosw.predict_proba(X_val)[:, 1]
xgb_nosw_metrics = evaluate(y_val, xgb_nosw_val_proba, "XGBoost sin scale_pos_weight (val)")

[0]	validation_0-auc:0.67995
[50]	validation_0-auc:0.74990
[100]	validation_0-auc:0.76155
[150]	validation_0-auc:0.76753
[200]	validation_0-auc:0.77077
[250]	validation_0-auc:0.77313
[300]	validation_0-auc:0.77492
[350]	validation_0-auc:0.77637
[400]	validation_0-auc:0.77721
[450]	validation_0-auc:0.77803
[500]	validation_0-auc:0.77877
[550]	validation_0-auc:0.77907
[600]	validation_0-auc:0.77950
[650]	validation_0-auc:0.77986
[700]	validation_0-auc:0.78015
[750]	validation_0-auc:0.78044
[800]	validation_0-auc:0.78074
[850]	validation_0-auc:0.78086
[900]	validation_0-auc:0.78095
[950]	validation_0-auc:0.78116
[999]	validation_0-auc:0.78129
Mejor iteración: 999
[XGBoost sin scale_pos_weight (val)] ROC-AUC=0.7813  PR-AUC=0.2803  KS=0.4249  Brier=0.0660


## Ajuste: más árboles para dejar converger el entrenamiento

Sin `scale_pos_weight` el Brier bajó de 0.1751 a 0.0660 (bien debajo del target ≤0.15) y el AUC mejoró levemente a 0.7813, confirmando la hipótesis: la reponderación distorsionaba la calibración sin aportar al ranking. Pero `best_iteration=999` (el tope de `n_estimators`) muestra que el entrenamiento no llegó a activar el early stopping: la curva de AUC en validación todavía subía al cortar. Subimos `n_estimators` a 3000 (manteniendo `early_stopping_rounds=50`) para dejar que el modelo converja de verdad antes de leer el resultado final, en vez de compararlo con un límite artificial.

In [7]:
xgb_params_final = dict(xgb_params_nosw)
xgb_params_final["n_estimators"] = 3000

xgb_model_final = xgb.XGBClassifier(**xgb_params_final)
xgb_model_final.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

print(f"Mejor iteración: {xgb_model_final.best_iteration}")

xgb_final_val_proba = xgb_model_final.predict_proba(X_val)[:, 1]
xgb_final_metrics = evaluate(y_val, xgb_final_val_proba, "XGBoost final (val)")

[0]	validation_0-auc:0.67995
[100]	validation_0-auc:0.76155
[200]	validation_0-auc:0.77077
[300]	validation_0-auc:0.77492
[400]	validation_0-auc:0.77721
[500]	validation_0-auc:0.77877
[600]	validation_0-auc:0.77950
[700]	validation_0-auc:0.78015
[800]	validation_0-auc:0.78074
[900]	validation_0-auc:0.78095
[1000]	validation_0-auc:0.78129
[1100]	validation_0-auc:0.78150
[1200]	validation_0-auc:0.78166
[1300]	validation_0-auc:0.78184
[1393]	validation_0-auc:0.78176
Mejor iteración: 1343
[XGBoost final (val)] ROC-AUC=0.7819  PR-AUC=0.2800  KS=0.4243  Brier=0.0660


## Ajuste: árboles más profundos

Con más rondas el modelo convergió de verdad (early stopping activado en la iteración 1393, mejor en 1343), pero la ganancia marginal por ronda ya era mínima: el AUC se estancó en ~0.782, todavía 0.008 debajo del target de 0.79. `max_depth=5` puede estar limitando cuánta interacción entre features puede capturar cada árbol (relevante acá porque muchas de las 301 features son agregaciones relacionales cuya señal combinada con `EXT_SOURCE_*` no es necesariamente lineal). Probamos `max_depth=7` con `learning_rate` más bajo (para compensar la mayor capacidad y no sobreajustar) y `min_child_weight` más bajo (los árboles más profundos generan hojas con menos observaciones, así que el mínimo por hoja debe bajar en proporción). Esta es la última pasada de tuneo manual contra este holdout: si no cierra la brecha, seguimos con la validación cruzada (paso 5) en vez de seguir ajustando contra un único split.

In [8]:
xgb_params_deep = dict(xgb_params_final)
xgb_params_deep["max_depth"] = 7
xgb_params_deep["learning_rate"] = 0.02
xgb_params_deep["min_child_weight"] = 10
xgb_params_deep["n_estimators"] = 4000

xgb_model_deep = xgb.XGBClassifier(**xgb_params_deep)
xgb_model_deep.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)

print(f"Mejor iteración: {xgb_model_deep.best_iteration}")

xgb_deep_val_proba = xgb_model_deep.predict_proba(X_val)[:, 1]
xgb_deep_metrics = evaluate(y_val, xgb_deep_val_proba, "XGBoost max_depth=7 (val)")

[0]	validation_0-auc:0.68922
[200]	validation_0-auc:0.77150
[400]	validation_0-auc:0.77797
[600]	validation_0-auc:0.78058
[800]	validation_0-auc:0.78130
[884]	validation_0-auc:0.78140
Mejor iteración: 834
[XGBoost max_depth=7 (val)] ROC-AUC=0.7815  PR-AUC=0.2808  KS=0.4250  Brier=0.0660


## Validación cruzada estratificada (5-fold)

Nos quedamos con la configuración de `max_depth=5` (`xgb_params_final`, sin `scale_pos_weight`): mismo AUC que `max_depth=7` pero más simple y con más margen antes del early stopping. Un único holdout (20% de train) puede sobre- o subestimar el AUC real por azar en cómo cayeron los casos difíciles; 5-fold CV promedia sobre 5 particiones distintas del set de train completo, dando una estimación más robusta y menos dependiente de un split particular. Cada fold entrena con su propio early stopping contra el fold de test correspondiente, usando la misma cantidad máxima de árboles (`best_iteration` de la corrida final, redondeado hacia arriba, como techo razonable en vez de repetir 3000 rondas completas 5 veces).

In [9]:
from sklearn.model_selection import StratifiedKFold

cv_params = dict(xgb_params_final)
cv_params["n_estimators"] = 2000

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    model = xgb.XGBClassifier(**cv_params)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

    proba = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, proba)
    cv_aucs.append(auc)
    print(f"Fold {fold}: AUC={auc:.4f} (best_iteration={model.best_iteration})")

print(f"\nCV AUC: {np.mean(cv_aucs):.4f} +/- {np.std(cv_aucs):.4f}")

Fold 0: AUC=0.7758 (best_iteration=978)
Fold 1: AUC=0.7848 (best_iteration=978)
Fold 2: AUC=0.7799 (best_iteration=1040)
Fold 3: AUC=0.7825 (best_iteration=1052)
Fold 4: AUC=0.7778 (best_iteration=1128)

CV AUC: 0.7802 +/- 0.0032


## Lectura del resultado de CV

`CV AUC: 0.7802 +/- 0.0032` confirma que el 0.7819 del holdout no fue un golpe de suerte de ese split particular: los 5 folds caen en un rango angosto (0.7758-0.7848), con la métrica del holdout dentro de ese rango. El AUC real del modelo está establemente en **~0.78**, por debajo del target de 0.79.

Esto es un resultado honesto a documentar, no a maquillar: superamos ampliamente el baseline (LogReg 0.7637 → XGBoost ~0.78, +1.6 puntos) y probamos que la brecha remanente no cede con tuneo manual de hiperparámetros (`scale_pos_weight`, cantidad de árboles, profundidad ya se probaron). Cerrar el 0.01 restante hasta 0.79 requeriría trabajo de otro tipo (más feature engineering, ensembles, optimización sistemática de hiperparámetros vía búsqueda), que queda fuera del alcance de tuneo manual de esta tarea y se puede señalar como próximo paso en el informe. Con XGBoost ya seleccionado como modelo primario, entrenamos la versión final sobre el 100% del set de train (sin holdout, ya no lo necesitamos para decidir hiperparámetros) para maximizar los datos disponibles antes de persistirlo.

In [13]:
FINAL_N_ESTIMATORS = int(round(np.mean([xgb_model_final.best_iteration] + [978, 978, 1040, 1052, 1128])))
print(f"n_estimators fijo para el modelo final (promedio de los best_iteration observados): {FINAL_N_ESTIMATORS}")

final_params = dict(xgb_params_final)
final_params["n_estimators"] = FINAL_N_ESTIMATORS
del final_params["early_stopping_rounds"]

xgb_final = xgb.XGBClassifier(**final_params)
xgb_final.fit(X, y, verbose=False)

print("Modelo final entrenado sobre el 100% del train.")

n_estimators fijo para el modelo final (promedio de los best_iteration observados): 1086
Modelo final entrenado sobre el 100% del train.


## Feature importance

Usamos `gain` (contribución promedio a la reducción de la función de pérdida por cada vez que la feature se usa en un split) en vez de `weight` (cantidad de splits), porque `gain` refleja mejor cuánto aporta cada feature al poder predictivo real y no solo cuán seguido aparece. Esto es insumo directo para la Tarea 5 (SHAP): adelanta qué features esperamos ver arriba del ranking de importancia global.

In [14]:
booster = xgb_final.get_booster()
gain_importance = booster.get_score(importance_type="gain")

importance_df = (
    pd.Series(gain_importance, name="gain")
    .sort_values(ascending=False)
    .head(20)
    .rename_axis("feature")
    .reset_index()
)
importance_df

,feature,gain
0,EXT_SOURCE_2,113.380081
1,EXT_SOURCE_3,101.618294
2,NAME_EDUCATION_TYPE_Higher education,80.318748
3,CODE_GENDER_M,58.108822
4,NAME_EDUCATION_TYPE_Secondary / secondary special,50.771797
5,NAME_INCOME_TYPE_Working,39.843906
6,EXT_SOURCE_1,37.808899
7,FLAG_OWN_CAR_N,37.427620
8,CODE_GENDER_F,34.371811
9,cc_utilization_mean,31.704363


**Lectura del top-20:** `EXT_SOURCE_2` y `EXT_SOURCE_3` encabezan, consistente con la Tarea 1 (eran las correlaciones más fuertes con `TARGET` en el EDA). El resto del top mezcla demografía (`NAME_EDUCATION_TYPE`, `CODE_GENDER`, `NAME_INCOME_TYPE`), variables de negocio (`credit_to_goods`) y varias agregaciones relacionales de la Tarea 2 (`cc_utilization_mean`, `prev_ever_refused`, `inst_late_rate`, `pos_credits_count`, `prev_approval_rate`, `bureau_overdue_sum`), lo que confirma que ese trabajo de feature engineering aporta señal real y no quedó fuera del modelo.

**Hallazgo a marcar para Tarea 5:** `CODE_GENDER_M` y `CODE_GENDER_F` aparecen en el puesto 4 y 9 del ranking, con gain comparable al de variables de negocio fuertes. A diferencia de auditar el género solo como atributo *proxy*, acá el género es una feature *directa* del modelo, lo que hace la auditoría de fairness de la Tarea 5 no solo recomendable sino necesaria: hay que verificar si el modelo está usando el género como proxy de otra señal correlacionada (ingreso, tipo de ocupación) o si introduce una disparidad de trato que haya que corregir con thresholding restringido por fairness.

## Tracking con MLflow

Usamos `mlflow-skinny` (el paquete `mlflow` completo todavía no soporta pandas 3.x, que es la versión que usa este proyecto). El backend de tracking basado en archivos (`file:../mlruns`) está en modo mantenimiento en las versiones recientes de MLflow, así que usamos SQLite (`sqlite:///../mlflow.db`, gitignoreado) como store, el backend recomendado. Registramos dos runs: el baseline (LogReg) y el modelo final (XGBoost), cada uno con sus hiperparámetros, las 4 métricas y el modelo serializado como artefacto (params, métricas, artefactos, con miras al model registry de auditabilidad).

In [15]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("credixai-tarea4-modelado")

with mlflow.start_run(run_name="baseline-logreg"):
    mlflow.log_params({"class_weight": "balanced", "max_iter": 1000})
    mlflow.log_metrics(logreg_metrics)
    mlflow.sklearn.log_model(
        logreg_pipeline, name="model",
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE,
    )

with mlflow.start_run(run_name="xgboost-final"):
    mlflow.log_params(final_params)
    mlflow.log_metrics(xgb_final_metrics)
    mlflow.log_metric("cv_auc_mean", float(np.mean(cv_aucs)))
    mlflow.log_metric("cv_auc_std", float(np.std(cv_aucs)))
    mlflow.xgboost.log_model(xgb_final, name="model")

print("Runs registrados en MLflow (sqlite:///../mlflow.db)")

2026/07/15 04:26:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/15 04:26:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/07/15 04:26:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Runs registrados en MLflow (sqlite:///../mlflow.db)
